# Задание 7

## Дифференциальная экспрессия генов

В данной работе проводится анализ RNA-seq данных с целью выявления генов, дифференциально экспрессированных между группами образцов.

В работе используются два подхода:

- DESeq2 - метод, основанный на отрицательном биномиальном распределении (подходит для count-based RNA-seq данных)
- LIMMA - линейная модель, адаптированная для RNA-seq через преобразование дисперсии

**Цель:**

- найти дифференциально экспрессированные гены
- оценить качество данных (PCA, clustering)
- сравнить методы анализа

**Данные**

raw counts table - ‘/home/STUDY/FBMF/bioinformatics/rna_seq_diff_exp/raw_expression_cells.tsv’ -
таблица с сырыми каунтами клеток из 4х разных клеточных линий колоректального рака. HCT-15, HCT-116, SW620 и SW837.

---

## Часть 1 — Анализ дифференциальной экспрессии (DESeq2)

1. Сделал `PCA` в ноутбуке, полученный результат представлен на графике ниже.

![photo](pictures/PCA.png)

**Вывод:** Видим наличие 4 выбросов (`HCT116`, `SW620`, `SW837D`, `SW837`), которые будут исключены из дальнейшего рассмотрения. Мы не имеем представления, почему они так распределяются, поэтому легче всего будет их просто убрать из анализа. 

В случае с `SW837D` и `SW837` скорее всего имеем дело с batch effect (систематическая техническая ошибка в данных, из-за которой образцы группируются не по биологии, а по условиям эксперимента) - похожие названия группируются отдельно ото всех других.

2. Построил clustermap образцов.

![photo](pictures/clustermap.png)

**Вывод:** Для оценки сходства между образцами была построена матрица попарных корреляций на основе экспрессии 2000 наиболее вариабельных генов по дисперсии. Использование только высоко-варьирующих генов позволяет снизить влияние шума от слабо экспрессируемых генов и повысить устойчивость кластеризации. Полученная clustermap показывает структурированное разделение образцов и согласуется с результатами PCA.

3. Провёл анализ дифференциально экспрессированных генов через скрипт на R.

4. Построил volcano plot для результатов дифференциальной экспрессии. 

![photo](pictures/Rplot.png)

5. **Вывод:** 

В рамках работы был проведён анализ RNA-seq данных с использованием DESeq2 для выявления дифференциально экспрессированных генов

На этапе провели предварительный анализ: PCA и clustermap. Обнаружили 4 образца, являющиеся выбросами скорее всего (`HCT116`, `SW620`, `SW837D`, `SW837`). Поскольку их происхождение неизвестно, они были исключены из дальнейшего анализа, чтобы избежать искажения результатов.

PCA и clustermap подтвердили наличие выраженной группировки образцов

В результате DESeq2-анализа была получена таблица дифференциально экспрессированных генов. Значимыми считались гены с:
- padj < 0.01
- |log2FC| > 3

Визуализировать распределение генов с помощью volcano plot.

Нашли набор наиболее биологически значимых кандидатов по уровню статистической значимости и величине изменения экспрессии



## Часть 2 — Анализ дифференциальной экспрессии (LIMMA)

---

В данной работе проводится анализ дифференциальной экспрессии генов с использованием метода `LIMMA`.  

**Цель исследования:** 

- выявить гены, статистически значимо различающиеся между исследуемыми группами образцов.

Для анализа использовались данные экспрессии генов, полученные методом микрочипирования. Для подобных данных метод `LIMMA` является одним из стандартных подходов, поскольку основан на линейных моделях и эмпирической байесовской оценке дисперсии, что особенно эффективно при небольшом числе образцов.

**В ходе работы были:**
- подготовлены данные для анализа;
- проведён анализ дифференциальной экспрессии;
- исследовано влияние различных порогов logFC;
- построены визуализации результатов;
- выполнено сравнение полученных результатов.

**Анализ дифференциальной экспрессии**
  
Для каждого гена были рассчитаны:
- logFC (log(fold change));
- p-value;
- adjusted p-value (FDR), но считали по обычному p-value, так как с adjusted ни одна точка не была выше трешхолдов (сделал это с разрешения преподавателя)

Значимыми считались гены, удовлетворяющие следующим условиям:
- p-value < 0.05;
- |logFC| выше заданного порога.

**Сравнение различных порогов logFC**

В работе были протестированы три значения порога:
- |logFC| > 1

![photo](pictures/logFC1.png)

- |logFC| > 2

![photo](pictures/logFC2.png)

- |logFC| > 3

![photo](pictures/logFC3.png)

Для каждого варианта было оценено количество значимых генов.

| Порог logFC | Количество значимых генов |
|--------------|---------------------------|
| 1            | 67                        |
| 2            |    4                      |
| 3            |     0                     |

При увеличении порога logFC количество значимых генов уменьшается, поскольку критерий становится более строгим.

- logFC = 1 позволяет выявить большее число генов, включая умеренные изменения экспрессии;
- logFC = 3 оставляет только наиболее выраженные изменения, однако может исключать биологически важные гены (в нашем случае пустое множество);
- logFC = 2 представляет компромисс между чувствительностью и строгостью отбора.

**Таблица результатов**

Для каждого гена были добавлены дополнительные столбцы в таблицу результатов `LIMMA_results.csv`:
- significance_logFC1
- significance_logFC2
- significance_logFC3

Они показывают, является ли ген значимым при соответствующем пороге logFC.

**Визуализация результатов**

Для каждого значения logFC были построены volcano plot, отображающие:
- величину изменения экспрессии (logFC);
- статистическую значимость (-log10 p-value).

Volcano plot позволяет визуально выделить:
- статистически значимые гены;
- гены с сильным изменением экспрессии;
- различия между выбранными порогами logFC.

**Интерпретация результатов**

При logFC = 1 наблюдается наибольшее количество значимых генов (67), однако часть из них может соответствовать слабым биологическим эффектам.

При logFC = 3 не было значимых генов, слишком строгий порог.

Для данного анализа наиболее оптимальным является порог logFC = 2, поскольку он позволяет сохранить баланс между количеством найденных генов и выраженностью биологического эффекта.

**Вывод:**

В ходе работы был проведён анализ дифференциальной экспрессии генов с использованием метода `LIMMA`.

Были исследованы различные значения порога logFC и показано, что увеличение порога приводит к уменьшению числа значимых генов.

Построенные volcano plot подтвердили различия между уровнями строгости отбора. Наиболее сбалансированным для данного набора данных оказался порог logFC = 2.

`LIMMA` был выбран для данного анализа, поскольку исследуемые данные представляют собой микрочиповые данные, для которых этот метод является одним из наиболее распространённых подходов.

Метод основан на линейных моделях и эмпирической байесовской оценке дисперсии, что позволяет получать более стабильные результаты даже при относительно небольшом числе образцов. Кроме того, `LIMMA` эффективно работает с высокоразмерными экспрессионными данными и предоставляет удобные инструменты для анализа дифференциальной экспрессии генов.
